In [ ]:
import itertools

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination



def build_bayesian_network():

    model = DiscreteBayesianNetwork([
        ("Confidence", "Correct"),
        ("Difficulty", "Correct"),
        ("Time", "Correct"),
        ("Hints", "Correct"),
        ("PreviousAccuracy", "Correct")
    ])

    # Confidence
    cpd_confidence = TabularCPD(
        "Confidence",
        3,
        [[0.33], [0.34], [0.33]],
        state_names={
            "Confidence": ["Low", "Medium", "High"]
        }
    )

    # Difficulty
    cpd_difficulty = TabularCPD(
        "Difficulty",
        3,
        [[0.34], [0.33], [0.33]],
        state_names={
            "Difficulty": ["Easy", "Medium", "Hard"]
        }
    )

    # Time
    cpd_time = TabularCPD(
        "Time",
        2,
        [[0.5], [0.5]],
        state_names={
            "Time": ["Fast", "Slow"]
        }
    )

    # Hints
    cpd_hints = TabularCPD(
        "Hints",
        2,
        [[0.7], [0.3]],
        state_names={
            "Hints": ["No", "Yes"]
        }
    )

    # Previous Accuracy
    cpd_prev = TabularCPD(
        "PreviousAccuracy",
        3,
        [[0.33], [0.34], [0.33]],
        state_names={
            "PreviousAccuracy": ["Low", "Medium", "High"]
        }
    )


    correct_probs = []
    incorrect_probs = []

    combos = itertools.product(
        ["Low", "Medium", "High"],       # Confidence
        ["Easy", "Medium", "Hard"],      # Difficulty
        ["Fast", "Slow"],                # Time
        ["No", "Yes"],                   # Hints
        ["Low", "Medium", "High"]        # Previous Accuracy
    )

    for conf, diff, time_, hint, prev in combos:

        score = 0.5

        # Confidence effect
        score += {
            "Low": -0.15,
            "Medium": 0.0,
            "High": 0.15
        }[conf]

        # Difficulty effect
        score += {
            "Easy": 0.20,
            "Medium": 0.0,
            "Hard": -0.20
        }[diff]

        # Time effect
        score += {
            "Fast": 0.05,
            "Slow": -0.05
        }[time_]

        # Hint effect
        score += {
            "No": 0.0,
            "Yes": -0.10
        }[hint]

        # Previous accuracy effect
        score += {
            "Low": -0.15,
            "Medium": 0.0,
            "High": 0.15
        }[prev]

        # Keep probability between 0.02 and 0.98
        score = min(max(score, 0.02), 0.98)

        correct_probs.append(score)
        incorrect_probs.append(1 - score)

 

    cpd_correct = TabularCPD(
        "Correct",
        2,
        [incorrect_probs, correct_probs],

        evidence=[
            "Confidence",
            "Difficulty",
            "Time",
            "Hints",
            "PreviousAccuracy"
        ],

        evidence_card=[
            3, 3, 2, 2, 3
        ],

        state_names={
            "Correct": ["No", "Yes"],
            "Confidence": ["Low", "Medium", "High"],
            "Difficulty": ["Easy", "Medium", "Hard"],
            "Time": ["Fast", "Slow"],
            "Hints": ["No", "Yes"],
            "PreviousAccuracy": ["Low", "Medium", "High"]
        }
    )

    # Add CPDs
    model.add_cpds(
        cpd_confidence,
        cpd_difficulty,
        cpd_time,
        cpd_hints,
        cpd_prev,
        cpd_correct
    )

    # Check network
    assert model.check_model()

    return model



def estimate_probability_correct(
    model,
    confidence,
    difficulty,
    time_,
    hints,
    previous_accuracy
):

    infer = VariableElimination(model)

    result = infer.query(
        variables=["Correct"],
        evidence={
            "Confidence": confidence,
            "Difficulty": difficulty,
            "Time": time_,
            "Hints": hints,
            "PreviousAccuracy": previous_accuracy
        }
    )

    return result.values[1]


if __name__ == "__main__":

    model = build_bayesian_network()

    print("=" * 60)
    print("           BAYESIAN NETWORK DEMO")
    print("=" * 60)

    # Given parameters
    confidence = "High"
    difficulty = "Hard"
    time_ = "Fast"
    hints = "No"
    previous_accuracy = "Low"

    probability = estimate_probability_correct(
        model,
        confidence,
        difficulty,
        time_,
        hints,
        previous_accuracy
    )

    print("\nLearner Parameters")
    print("-" * 40)

    print("Confidence        :", confidence)
    print("Difficulty        :", difficulty)
    print("Time              :", time_)
    print("Hints             :", hints)
    print("Previous Accuracy :", previous_accuracy)

    print("\nPrediction")
    print("-" * 40)

    print(f"P(Correct)   = {probability:.2f}")
    print(f"P(Incorrect) = {1 - probability:.2f}")

    if probability >= 0.5:
        print("\nPrediction: Student is likely to answer CORRECTLY")
    else:
        print("\nPrediction: Student is likely to answer INCORRECTLY")

C:\Users\Rupanshi\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Rupanshi\AppData\Roaming\Python\Python314\site-packages\pgmpy\estimators\__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


           BAYESIAN NETWORK DEMO

Learner Parameters
----------------------------------------
Confidence        : High
Difficulty        : Hard
Time              : Fast
Hints             : No
Previous Accuracy : Low

Prediction
----------------------------------------
P(Correct)   = 0.35
P(Incorrect) = 0.65

Prediction: Student is likely to answer INCORRECTLY
